<a href="https://colab.research.google.com/github/shiyuto6/STAD68Project/blob/main/STAD68_Project_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Pythia-1.4B CE VS KD Downstream Tasks Comparison
Prune 1.4B model to 1B

### Set Up

In [ ]:
!pip -q install transformers datasets accelerate evaluate lm-eval==0.4.3

import math, json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    set_seed,
    get_linear_schedule_with_warmup
)

In [ ]:
# Config
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

set_seed(123)

MODEL_NAME = "EleutherAI/pythia-1.4b"
DTYPE = torch.bfloat16

BLOCK_SIZE = 256

CALIB_SAMPLES = 1024
TRAIN_SAMPLES = 100000
EVAL_SAMPLES  = 5000

BATCH_SIZE = 16
GRAD_ACCUM = 4

PRUNE_RATIO = 0.40

CE_STEPS = 2500
CE_LR = 5e-6
CE_WARMUP = 200
CE_CLIP = 0.5

KD_STEPS = 2500
KD_LR = 5e-6
KD_WARMUP = 200
KD_CLIP = 0.5
KD_TAU = 1.0

HELLASWAG_FEWSHOT = 5

DEVICE: cuda


In [ ]:
def load_pythia(model_name=MODEL_NAME):
    m = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=DTYPE).to(DEVICE)
    m.config.use_cache = False
    return m

In [ ]:
import os
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader
from google.colab import drive

drive.mount('/content/drive')

TRAIN_CACHE = "/content/drive/MyDrive/owt_train_blocked_256"
VAL_CACHE   = "/content/drive/MyDrive/owt_val_blocked_256"

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

if os.path.exists(TRAIN_CACHE) and os.path.exists(VAL_CACHE):
    print("Cache found on Drive, loading...")
    lm_train = load_from_disk(TRAIN_CACHE)
    lm_val   = load_from_disk(VAL_CACHE)
    print(f"Loaded. train: {len(lm_train)} rows, val: {len(lm_val)} rows")

else:
    print("No cache found, tokenizing from scratch...")

    raw = load_dataset("openwebtext")
    split = raw["train"].train_test_split(test_size=0.005, seed=123)

    raw_train = split["train"].select(range(min(80000, len(split["train"]))))
    raw_val   = split["test"].select(range(min(8000,  len(split["test"]))))

    def tokenize_fn(examples):
        return tok(examples["text"], return_special_tokens_mask=False)

    tokenized_train = raw_train.map(tokenize_fn, batched=True, remove_columns=["text"], num_proc=2)
    tokenized_val   = raw_val.map(tokenize_fn,   batched=True, remove_columns=["text"], num_proc=2)

    def group_texts(examples):
        concatenated = {k: sum(examples[k], []) for k in examples.keys()}
        total_len = (len(concatenated["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE
        result = {}
        for k, v in concatenated.items():
            v = v[:total_len]
            result[k] = [v[i:i + BLOCK_SIZE] for i in range(0, total_len, BLOCK_SIZE)]
        result["labels"] = result["input_ids"].copy()
        return result

    lm_train = tokenized_train.map(group_texts, batched=True, num_proc=2)
    lm_val   = tokenized_val.map(group_texts,   batched=True, num_proc=2)

    print(f"Tokenized. train: {len(lm_train)} rows, val: {len(lm_val)} rows")
    print("Saving to Google Drive...")
    lm_train.save_to_disk(TRAIN_CACHE)
    lm_val.save_to_disk(VAL_CACHE)
    print("Saved. Future sessions will load from cache automatically.")

def take_subset(ds, n):
    return ds.select(range(min(n, len(ds))))

train_ds = take_subset(lm_train, TRAIN_SAMPLES)
val_ds   = take_subset(lm_val,   EVAL_SAMPLES)
test_ds  = take_subset(lm_val,   EVAL_SAMPLES)
calib_ds = take_subset(lm_train, CALIB_SAMPLES)

collator = DataCollatorForLanguageModeling(tok, mlm=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collator)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
calib_loader = DataLoader(calib_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

print("train:", len(train_ds), "val:", len(val_ds), "test:", len(test_ds), "calib:", len(calib_ds))

In [ ]:
# Functions debugging Nan PPL
def count_params(model):
    return sum(p.numel() for p in model.parameters())

@torch.no_grad()
def eval_ppl(model, loader, max_batches=None):
    model.eval()
    total_loss, total_tokens = 0.0, 0

    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break

        batch = {k: v.to(DEVICE) for k, v in batch.items() if torch.is_tensor(v)}
        out = model(**batch, use_cache=False)

        labels = batch.get("labels", None)
        if labels is None:
            bs, seqlen = batch["input_ids"].shape
            n_valid = bs * (seqlen - 1)
        else:
            n_valid = (labels != -100).sum().item()

        total_loss += out.loss.item() * n_valid
        total_tokens += n_valid

    mean_loss = total_loss / max(1, total_tokens)
    return math.exp(mean_loss), mean_loss

@torch.no_grad()
def quick_logit_check(model, loader):
    model.eval()
    batch = next(iter(loader))
    batch = {k: v.to(DEVICE) for k,v in batch.items() if torch.is_tensor(v)}
    out = model(**batch, use_cache=False)
    print("logits_nan:", torch.isnan(out.logits).any().item(),
          "logits_inf:", torch.isinf(out.logits).any().item(),
          "loss:", out.loss.item())

def find_nan_params(model, max_print=20):
    bad = []
    for name, p in model.named_parameters():
        if torch.isnan(p).any() or torch.isinf(p).any():
            bad.append(name)
            if len(bad) >= max_print:
                break
    print("Bad params:", bad if bad else "None")


### Functions for computing importance score

In [ ]:
def get_pythia_layers(model):
    return model.gpt_neox.layers

def compute_neuron_importance(model, loader, max_batches=None):
    model.eval()
    layers = get_pythia_layers(model)
    scores = [None for _ in layers]
    hooks = []

    def make_hook(i):
        def hook_fn(module, inp, out):
            x = out.detach().float()
            l2_over_batch = torch.sqrt((x ** 2).sum(dim=0) + 1e-12)
            s = l2_over_batch.mean(dim=0)
            if scores[i] is None:
                scores[i] = s
            else:
                scores[i] += s
        return hook_fn

    for i, layer in enumerate(layers):
        hooks.append(layer.mlp.act.register_forward_hook(make_hook(i)))

    with torch.no_grad():
        for bi, batch in enumerate(loader):
            if max_batches is not None and bi >= max_batches:
                break
            batch = {k: v.to(DEVICE) for k,v in batch.items() if torch.is_tensor(v)}
            _ = model(**batch)

    for h in hooks:
        h.remove()

    for i in range(len(scores)):
        if scores[i] is None:
            raise RuntimeError("No activation captured; hook failed.")
    return scores

def prune_pythia_mlp_width(model, keep_ratio, importance_list):
    layers = get_pythia_layers(model)

    for i, layer in enumerate(layers):
        mlp = layer.mlp
        old_in  = mlp.dense_h_to_4h
        old_out = mlp.dense_4h_to_h

        dev = old_in.weight.device
        dt  = old_in.weight.dtype

        imp = importance_list[i].to(device=dev, dtype=torch.float32)
        inter_dim = imp.numel()
        keep_k = max(1, int(round(keep_ratio * inter_dim)))

        keep_idx = torch.topk(imp, k=keep_k, largest=True).indices
        keep_idx, _ = torch.sort(keep_idx)

        hidden = old_in.in_features

        new_in  = nn.Linear(hidden, keep_k, bias=(old_in.bias is not None)).to(device=dev, dtype=dt)
        new_out = nn.Linear(keep_k, hidden, bias=(old_out.bias is not None)).to(device=dev, dtype=dt)

        new_in.weight.data.copy_(old_in.weight.data[keep_idx, :].contiguous())
        if old_in.bias is not None:
            new_in.bias.data.copy_(old_in.bias.data[keep_idx].contiguous())

        new_out.weight.data.copy_(old_out.weight.data[:, keep_idx].contiguous())
        if old_out.bias is not None:
            new_out.bias.data.copy_(old_out.bias.data.contiguous())

        mlp.dense_h_to_4h = new_in
        mlp.dense_4h_to_h = new_out

    model.config.use_cache = False
    return model



### Conventional Training

In [ ]:
def freeze_embeddings_and_head(model, freeze=True):
    req = not freeze
    for p in model.gpt_neox.embed_in.parameters():
        p.requires_grad = req
    for p in model.embed_out.parameters():
        p.requires_grad = req

def train_ce(model, train_loader, val_loader,
                          steps=400, lr=5e-6, warmup_steps=100, clip=0.5,
                          weight_decay=0.01, log_every=50, freeze_io=True,
                          grad_accum=1):
    model.train()
    if freeze_io:
        freeze_embeddings_and_head(model, freeze=True)

    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay
    )
    sched = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=warmup_steps, num_training_steps=steps
    )

    it = iter(train_loader)

    for step in range(steps):
        model.train()
        opt.zero_grad(set_to_none=True)

        accum_loss = 0.0
        valid_accum_steps = 0

        for accum_step in range(grad_accum):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(train_loader)
                batch = next(it)

            batch = {k: v.to(DEVICE) for k, v in batch.items() if torch.is_tensor(v)}
            out = model(**batch)
            loss = out.loss / grad_accum

            if not torch.isfinite(loss):
                print(f"Non-finite loss at step {step+1}, accum {accum_step}, skipping")
                continue

            loss.backward()
            accum_loss += loss.item()
            valid_accum_steps += 1

        if valid_accum_steps == 0:
            print(f"All accum steps non-finite at step {step+1}, skipping optimizer step")
            opt.zero_grad(set_to_none=True)
            continue

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        if not torch.isfinite(grad_norm):
            print(f"Non-finite grad norm at step {step+1}, skipping optimizer step")
            opt.zero_grad(set_to_none=True)
            continue

        opt.step()
        sched.step()

        if (step + 1) % log_every == 0 or step == 0:
            model.eval()
            with torch.no_grad():
                b2 = next(iter(val_loader))
                b2 = {k: v.to(DEVICE) for k, v in b2.items() if torch.is_tensor(v)}
                out2 = model(**b2)
                print("After update check:",
                      "logits_nan", torch.isnan(out2.logits).any().item(),
                      "loss", out2.loss.item())
            ppl, _ = eval_ppl(model, val_loader, max_batches=50)
            print(f"[CE] step {step+1}/{steps}  "
                  f"train_loss={accum_loss:.4f}  val_ppl={ppl:.2f}  "
                  f"grad_norm={grad_norm:.3f}")

    if freeze_io:
        freeze_embeddings_and_head(model, freeze=False)

### KD retraining

In [ ]:
def train_kd(
    student, teacher, train_loader, val_loader,
    steps=200, lr=5e-6, warmup_steps=100, clip=0.5,
    tau=1.0, weight_decay=0.01, log_every=50, freeze_io=True,
    grad_accum=1
):
    teacher.eval()
    student.train()

    if freeze_io:
        freeze_embeddings_and_head(student, freeze=True)

    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, student.parameters()),
        lr=lr,
        weight_decay=weight_decay
    )
    sched = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=warmup_steps, num_training_steps=steps
    )

    it = iter(train_loader)
    T = float(tau)

    for step in range(steps):
        student.train()
        opt.zero_grad(set_to_none=True)

        accum_loss = 0.0
        valid_accum_steps = 0

        for accum_step in range(grad_accum):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(train_loader)
                batch = next(it)

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch.get("attention_mask", None)
            if attention_mask is None:
                attention_mask = torch.ones_like(input_ids, dtype=torch.float32, device=DEVICE)
            else:
                attention_mask = attention_mask.to(DEVICE).float()

            # Teacher forward
            with torch.no_grad():
                t_logits = teacher(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).logits

            # Student forward
            s_logits = student(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).logits

            t_logits = t_logits[:, :-1, :].float()
            s_logits = s_logits[:, :-1, :].float()
            mask = attention_mask[:, 1:]

            t_logp = F.log_softmax(t_logits / T, dim=-1)
            s_logp = F.log_softmax(s_logits / T, dim=-1)

            t_prob = t_logp.exp()
            kl_tok = (t_prob * (t_logp - s_logp)).sum(dim=-1)

            denom = mask.sum().clamp_min(1.0)
            loss = (kl_tok * mask).sum() / denom
            loss = loss * (T * T)
            loss = loss / grad_accum

            if not torch.isfinite(loss):
                print(f"Non-finite KD loss at step {step+1}, accum {accum_step}, skipping")
                continue

            loss.backward()
            accum_loss += loss.item()
            valid_accum_steps += 1

        if valid_accum_steps == 0:
            print(f"All accum steps non-finite at step {step+1}, skipping optimizer step")
            opt.zero_grad(set_to_none=True)
            continue

        grad_norm = torch.nn.utils.clip_grad_norm_(student.parameters(), clip)
        if not torch.isfinite(grad_norm):
            print(f"Non-finite grad norm at step {step+1}, skipping optimizer step")
            opt.zero_grad(set_to_none=True)
            continue

        opt.step()
        sched.step()

        if (step + 1) % log_every == 0 or step == 0:
            student.eval()
            with torch.no_grad():
                b2 = next(iter(val_loader))
                b2 = {k: v.to(DEVICE) for k, v in b2.items() if torch.is_tensor(v)}
                out2 = student(**b2, use_cache=False)
                print("After update check:",
                      "logits_nan", torch.isnan(out2.logits).any().item(),
                      "loss", out2.loss.item())
            ppl, _ = eval_ppl(student, val_loader, max_batches=50)
            print(f"[KD] step {step+1}/{steps}  "
                  f"kd_loss={accum_loss:.4f}  val_ppl={ppl:.2f}  "
                  f"grad_norm={grad_norm:.3f}")

    if freeze_io:
        freeze_embeddings_and_head(student, freeze=False)

In [ ]:
def summarize_model_size(model, model_name="model"):
    total_params = 0
    trainable_params = 0

    for p in model.parameters():
        num = p.numel()
        total_params += num
        if p.requires_grad:
            trainable_params += num

    print(f"===== {model_name} =====")
    print(f"Total parameters:     {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Non-trainable params: {total_params - trainable_params:,}")
    print(f"Trainable ratio:      {100 * trainable_params / total_params:.2f}%")
    print()

    return total_params, trainable_params

### Building Models
## Baseline

In [ ]:
teacher = load_pythia()
teacher.config.use_cache = False
teacher.eval()
print("Teacher params:", count_params(teacher))
ppl_val, _ = eval_ppl(teacher, val_loader)
ppl_test, _ = eval_ppl(teacher, test_loader)
print(f"[Baseline] val_ppl={ppl_val:.2f} test_ppl={ppl_test:.2f}")

## Pruned without retraining

In [ ]:
student0 = load_pythia()
imp = compute_neuron_importance(student0, calib_loader)

student_pruned = prune_pythia_mlp_width(student0, keep_ratio=1.0-PRUNE_RATIO, importance_list=imp).to(DEVICE)
print("Pruned params:", count_params(student_pruned))
quick_logit_check(student_pruned, val_loader)
find_nan_params(student_pruned)

ppl_val, _ = eval_ppl(student_pruned, val_loader)
ppl_test, _ = eval_ppl(student_pruned, test_loader)
print(f"[Pruned no retrain] val_ppl={ppl_val:.2f} test_ppl={ppl_test:.2f}")


## Pruned + KD retraining

In [ ]:
student_kd = load_pythia()
imp_kd = compute_neuron_importance(student_kd, calib_loader)
student_kd = prune_pythia_mlp_width(student_kd, keep_ratio=1.0-PRUNE_RATIO, importance_list=imp_kd).to(DEVICE)

print("KD model sanity check before training:")
quick_logit_check(student_kd, val_loader)

train_kd(
    student_kd, teacher, train_loader, val_loader,
    steps=KD_STEPS, lr=KD_LR, warmup_steps=KD_WARMUP, clip=KD_CLIP,
    tau=KD_TAU, log_every=50, freeze_io=True,
    grad_accum=GRAD_ACCUM
)

ppl_val, _ = eval_ppl(student_kd, val_loader)
ppl_test, _ = eval_ppl(student_kd, test_loader)
print(f"[KD final] val_ppl={ppl_val:.2f} test_ppl={ppl_test:.2f}")

## Pruned + conventional retraining

In [ ]:
student_ce = load_pythia()
imp_ce = compute_neuron_importance(student_ce, calib_loader)
student_ce = prune_pythia_mlp_width(student_ce, keep_ratio=1.0-PRUNE_RATIO, importance_list=imp_ce).to(DEVICE)

print("CE model sanity check before training:")
quick_logit_check(student_ce, val_loader)

train_ce(
    student_ce, train_loader, val_loader,
    steps=CE_STEPS, lr=CE_LR, warmup_steps=CE_WARMUP, clip=CE_CLIP,
    log_every=50, freeze_io=True,
    grad_accum=GRAD_ACCUM
)

ppl_val, _ = eval_ppl(student_ce, val_loader)
ppl_test, _ = eval_ppl(student_ce, test_loader)
print(f"[CE final] val_ppl={ppl_val:.2f} test_ppl={ppl_test:.2f}")

### Save Models and Evaluate

In [ ]:
tok.save_pretrained("/content/tokenizer")
teacher.save_pretrained("/content/model_base")
student_ce.save_pretrained("/content/model_pruned_ce")
student_kd.save_pretrained("/content/model_pruned_kd")

print("Saved models to /content/model_base, /content/model_pruned_ce, /content/model_pruned_kd")

In [ ]:
# !python -m lm_eval \
#   --model hf \
#   --model_args pretrained=/content/model_base,tokenizer=/content/tokenizer \
#   --tasks hellaswag \
#   --device cuda \
#   --batch_size 8 \
#   --num_fewshot 5 \
#   --output_path /content/hellaswag_base.json

In [ ]:
def set_pythia_intermediate_size_in_config(model):
    keep_k = model.gpt_neox.layers[0].mlp.dense_h_to_4h.out_features
    model.config.intermediate_size = int(keep_k)
    return keep_k

In [ ]:
keep_k_ce = set_pythia_intermediate_size_in_config(student_ce)
student_ce.save_pretrained("/content/model_pruned_ce")
tok.save_pretrained("/content/tokenizer")
print("Saved CE with intermediate_size =", keep_k_ce)

keep_k_kd = set_pythia_intermediate_size_in_config(student_kd)
student_kd.save_pretrained("/content/model_pruned_kd")
print("Saved KD with intermediate_size =", keep_k_kd)


In [ ]:
!lm_eval \
  --model hf \
  --model_args pretrained=/content/model_pruned_ce,tokenizer=/content/tokenizer \
  --tasks hellaswag \
  --device cuda \
  --batch_size 8 \
  --num_fewshot 5 \
  --output_path /content/hellaswag_pruned_ce.json

2026-03-17:03:17:52,750 INFO     [__main__.py:272] Verbosity set to INFO
2026-03-17:03:17:57,495 INFO     [__main__.py:369] Selected Tasks: ['hellaswag']
2026-03-17:03:17:57,497 INFO     [evaluator.py:152] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
2026-03-17:03:17:57,497 INFO     [evaluator.py:189] Initializing hf model, with arguments: {'pretrained': '/content/model_pruned_ce', 'tokenizer': '/content/tokenizer'}
2026-03-17:03:17:57,537 INFO     [huggingface.py:170] Using device 'cuda'
Loading weights: 100% 292/292 [00:00<00:00, 487.57it/s, Materializing param=gpt_neox.layers.23.post_attention_layernorm.weight]
2026-03-17:03:17:59,232 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/datasets/hellaswag/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-17:03:17:59,469 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/datasets/Rowan/hellaswag/resolve/main/README.md "HTTP/1.1 307 Temporary

In [ ]:
!lm_eval \
  --model hf \
  --model_args pretrained=/content/model_pruned_kd,tokenizer=/content/tokenizer \
  --tasks hellaswag \
  --device cuda \
  --batch_size 8 \
  --num_fewshot 5 \
  --output_path /content/hellaswag_pruned_kd.json

2026-03-17:03:05:18,177 INFO     [__main__.py:272] Verbosity set to INFO
2026-03-17:03:05:23,444 INFO     [__main__.py:369] Selected Tasks: ['hellaswag']
2026-03-17:03:05:23,446 INFO     [evaluator.py:152] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
2026-03-17:03:05:23,446 INFO     [evaluator.py:189] Initializing hf model, with arguments: {'pretrained': '/content/model_pruned_kd', 'tokenizer': '/content/tokenizer'}
2026-03-17:03:05:23,484 INFO     [huggingface.py:170] Using device 'cuda'
Loading weights: 100% 292/292 [00:00<00:00, 493.57it/s, Materializing param=gpt_neox.layers.23.post_attention_layernorm.weight]
2026-03-17:03:05:25,150 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/datasets/hellaswag/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-17:03:05:25,379 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/datasets/Rowan/hellaswag/resolve/main/README.md "HTTP/1.1 307 Temporary

In [ ]:
pythia_1b = AutoModelForCausalLM.from_pretrained(
    "EleutherAI/pythia-1b",
    torch_dtype=DTYPE,
).to(DEVICE)
pythia_1b.config.use_cache = False
pythia_1b.eval()

print("Pythia-1b params:", count_params(pythia_1b))
ppl_val, _  = eval_ppl(pythia_1b, val_loader)
ppl_test, _ = eval_ppl(pythia_1b, test_loader)
print(f"[Pythia-1b baseline] val_ppl={ppl_val:.2f} test_ppl={ppl_test:.2f}")

pythia_1b.save_pretrained("/content/model_pythia1b_baseline")
tok_1b = AutoTokenizer.from_pretrained("EleutherAI/pythia-1b", use_fast=True)
tok_1b.save_pretrained("/content/tokenizer_1b")

del pythia_1b
torch.cuda.empty_cache()

In [ ]:
!lm_eval \
  --model hf \
  --model_args pretrained=/content/model_pythia1b_baseline,tokenizer=/content/tokenizer_1b \
  --tasks hellaswag \
  --device cuda \
  --batch_size 8 \
  --num_fewshot 5 \
  --output_path /content/hellaswag_pythia1b_baseline.json


In [ ]:
teacher1, _ = summarize_model_size(teacher, "Base Pythia-1.4B")
student_ce1, _ = summarize_model_size(student_ce, "CE Pythia-1.4B")
student_kd1, _ = summarize_model_size(student_kd, "KD Pythia-1.4B")